# Notebook 4: New Research Angles

Exploratory analysis beyond Begenau et al. — correlation mining, subgroup analysis, and patterns that could motivate extensions or new research questions.

**Sections:**
1. Consultant effects on allocation
2. Funding pressure and risk-taking (lagged panel)
3. Return chasing
4. Herd behavior / convergence (Galton regression)
5. Size tiers (small/medium/large)
6. Fiscal year timing effects
7. Target-actual divergence patterns
8. Maturity and risk appetite (interaction with funding)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

COLORS = {
    'blue': '#1f77b4', 'orange': '#d35400', 'purple': '#7f6d9e',
    'light_blue': '#aec7e8', 'red': '#d62728', 'green': '#2ca02c',
    'teal': '#17becf', 'brown': '#8c564b',
}

OUTPUT_DIR = Path('../../output_python/exploration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
df = pd.read_csv('../../output_python/ppd_plan_level_clean.csv')
perf = pd.read_csv('../../output_python/ppd_performance.csv')

# Derived variables
df['alternatives_trgt'] = (
    df['PETotal_Trgt'].fillna(0) + df['RETotal_Trgt'].fillna(0)
    + df['COMDTotal_Trgt'].fillna(0) + df['HFTotal_Trgt'].fillna(0)
    + df['AltMiscTotal_Trgt'].fillna(0) + df['OtherTotal_Trgt'].fillna(0)
)
df['public_equity_trgt'] = df['EQTotal_Trgt'].fillna(0)
df['fixed_income_trgt'] = df['FITotal_Trgt'].fillna(0) + df['CashTotal_Trgt'].fillna(0)
df['risky_trgt'] = df['public_equity_trgt'] + df['alternatives_trgt']
df['risky_actl'] = (df['EQTotal_Actl'].fillna(0) + df['PETotal_Actl'].fillna(0)
    + df['RETotal_Actl'].fillna(0) + df['COMDTotal_Actl'].fillna(0)
    + df['HFTotal_Actl'].fillna(0) + df['AltMiscTotal_Actl'].fillna(0)
    + df['OtherTotal_Actl'].fillna(0))
denom = df['alternatives_trgt'] + df['public_equity_trgt']
df['omega'] = np.where(denom > 0, df['alternatives_trgt'] / denom, np.nan)
df['pct_retired'] = np.where(df['TotMembership'] > 0, df['beneficiaries_tot'] / df['TotMembership'], np.nan)

# System-level aggregation
def weighted_avg(group, col, weight_col='MktAssets_net'):
    mask = group[col].notna() & group[weight_col].notna() & (group[weight_col] > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(group.loc[mask, col], weights=group.loc[mask, weight_col])

agg_cols = [
    'alternatives_trgt', 'public_equity_trgt', 'fixed_income_trgt',
    'risky_trgt', 'risky_actl', 'omega',
    'ActFundedRatio_GASB', 'InvestmentReturnAssumption_GASB',
    'InvestmentReturn_1yr', 'ret_bgnassets', 'pct_retired',
    'CashTotal_Trgt',
]

rows = []
for (pub_id, fy), grp in df.groupby(['pub_id', 'fy']):
    row = {'pub_id': pub_id, 'fy': fy}
    for col in agg_cols:
        if col in grp.columns:
            row[col] = weighted_avg(grp, col)
    row['MktAssets_net'] = grp['MktAssets_net'].sum()
    row['TotMembership'] = grp['TotMembership'].sum()
    row['beneficiaries_tot'] = grp['beneficiaries_tot'].sum()
    for col in ['pub_system_name', 'StateAbbrev', 'region', 'general_consultant', 'consultant_id']:
        if col in grp.columns:
            row[col] = grp[col].iloc[0]
    rows.append(row)

sys_df = pd.DataFrame(rows)
sys_df['pct_retired'] = np.where(sys_df['TotMembership'] > 0, sys_df['beneficiaries_tot'] / sys_df['TotMembership'], np.nan)

# Merge performance data
sys_df = sys_df.merge(perf[['pub_id', 'fy', 'ret_bgnassets', 'pfmonth']].rename(
    columns={'ret_bgnassets': 'ret_perf'}), on=['pub_id', 'fy'], how='left')
# Use perf file ret_bgnassets where available
sys_df['ret_bgnassets'] = sys_df['ret_perf'].fillna(sys_df['ret_bgnassets'])

print(f'System-level: {sys_df.shape[0]} obs, {sys_df["pub_id"].nunique()} systems')

## Section 1: Consultant Effects

Do systems with the same consultant have more similar allocations?

In [ ]:
# Consultant effects on omega
sys_2021 = sys_df[sys_df['fy'] == 2021].dropna(subset=['omega', 'general_consultant']).copy()
sys_2021['omega_pct'] = sys_2021['omega'] * 100

# Top consultants by number of clients
consultant_counts = sys_2021['general_consultant'].value_counts()
top_consultants = consultant_counts[consultant_counts >= 3].index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Box plot by top consultants
ax = axes[0]
cons_data = sys_2021[sys_2021['general_consultant'].isin(top_consultants)].copy()
cons_order = cons_data.groupby('general_consultant')['omega_pct'].median().sort_values(ascending=False).index
sns.boxplot(data=cons_data, x='general_consultant', y='omega_pct',
            order=cons_order, color=COLORS['light_blue'], ax=ax, fliersize=3)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('Omega (%)')
ax.set_title(f'Omega by Consultant (FY2021, consultants with 3+ clients)')
ax.grid(axis='y', alpha=0.3)

# Right: Within-consultant vs across-consultant variance
# ANOVA: is consultant a significant grouper?
groups = [g['omega_pct'].values for _, g in cons_data.groupby('general_consultant') if len(g) >= 3]
if len(groups) >= 2:
    f_stat, p_val = stats.f_oneway(*groups)
    
    # Within vs between variance
    grand_mean = cons_data['omega_pct'].mean()
    between_var = sum(len(g) * (g.mean() - grand_mean)**2 for g in [pd.Series(x) for x in groups]) / (len(groups) - 1)
    within_var = sum(g.var() * (len(g) - 1) for g in [pd.Series(x) for x in groups]) / (sum(len(g) for g in groups) - len(groups))
    
    ax = axes[1]
    ax.bar(['Between Consultants', 'Within Consultant'], [between_var, within_var],
           color=[COLORS['blue'], COLORS['orange']], alpha=0.85)
    ax.set_ylabel('Variance of Omega (pp²)')
    ax.set_title(f'Variance Decomposition\nF={f_stat:.2f}, p={p_val:.4f}')
    ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_consultant_effects.png')
plt.show()

# Print consultant stats
print(f'Consultants with 3+ clients: {len(top_consultants)}')
print(f'ANOVA F-stat: {f_stat:.2f}, p-value: {p_val:.4f}')
print(f'\nConsultant client counts (top 10):')
print(consultant_counts.head(10).to_string())

## Section 2: Funding Pressure and Risk-Taking

Does a drop in funding ratio predict a subsequent increase in alternatives? Lagged panel specification.

In [ ]:
# Lagged panel: delta_omega_t on funding_{t-1}
# Create lagged variables
sys_panel = sys_df.sort_values(['pub_id', 'fy']).copy()
sys_panel['omega_lag1'] = sys_panel.groupby('pub_id')['omega'].shift(1)
sys_panel['funding_lag1'] = sys_panel.groupby('pub_id')['ActFundedRatio_GASB'].shift(1)
sys_panel['delta_omega'] = sys_panel['omega'] - sys_panel['omega_lag1']
sys_panel['delta_funding'] = sys_panel['ActFundedRatio_GASB'] - sys_panel['funding_lag1']

# Regression: delta_omega_t = alpha + beta * funding_{t-1} + epsilon
reg_data = sys_panel.dropna(subset=['delta_omega', 'funding_lag1', 'StateAbbrev']).copy()
y = reg_data['delta_omega']
X = sm.add_constant(reg_data['funding_lag1'])
model_lag = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': reg_data['StateAbbrev']})

# Binscatter
def binscatter_simple(x, y, n_bins=20, ax=None, xlabel='', ylabel='', title='', color=None):
    data = pd.DataFrame({'x': x, 'y': y}).dropna()
    data['bin'] = pd.qcut(data['x'], n_bins, labels=False, duplicates='drop')
    binned = data.groupby('bin')[['x', 'y']].mean()
    X_ols = sm.add_constant(data['x'])
    model = sm.OLS(data['y'], X_ols).fit(cov_type='HC1')
    c = color or COLORS['blue']
    ax.scatter(binned['x'] * 100, binned['y'] * 100, s=60, color=c, zorder=5)
    x_line = np.linspace(data['x'].min(), data['x'].max(), 100)
    y_line = model.params['const'] + model.params['x'] * x_line
    ax.plot(x_line * 100, y_line * 100, '--', color=COLORS['red'], lw=1.5, alpha=0.8)
    ax.text(0.05, 0.95, f'beta={model.params["x"]:.4f}\nt={model.tvalues["x"]:.2f}\nR²={model.rsquared:.3f}\nN={int(model.nobs)}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    return model

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Lagged funding level -> delta omega
binscatter_simple(reg_data['funding_lag1'], reg_data['delta_omega'],
                  ax=axes[0],
                  xlabel='Funding Ratio_{t-1} (%)',
                  ylabel='Delta Omega_t (pp)',
                  title='Funding Pressure: Does Low Funding Predict More Alternatives?')

# Right: Change in funding -> delta omega
reg_data2 = sys_panel.dropna(subset=['delta_omega', 'delta_funding']).copy()
binscatter_simple(reg_data2['delta_funding'], reg_data2['delta_omega'],
                  ax=axes[1],
                  xlabel='Delta Funding Ratio (pp)',
                  ylabel='Delta Omega (pp)',
                  title='Concurrent: Funding Decline & Alternatives Increase?')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_funding_pressure.png')
plt.show()

print(f'\nLagged panel regression (state-clustered SE):')
print(f'  beta = {model_lag.params["funding_lag1"]:.4f}')
print(f'  t-stat = {model_lag.tvalues["funding_lag1"]:.2f}')
print(f'  N = {int(model_lag.nobs)}')

## Section 3: Return Chasing

Do pensions increase alternatives allocation after good past returns?

In [ ]:
# Return chasing: lagged return -> delta_omega
sys_panel['ret_lag1'] = sys_panel.groupby('pub_id')['ret_bgnassets'].shift(1)

reg_ret = sys_panel.dropna(subset=['delta_omega', 'ret_lag1']).copy()

fig, ax = plt.subplots(figsize=(8, 5))
binscatter_simple(reg_ret['ret_lag1'], reg_ret['delta_omega'],
                  ax=ax,
                  xlabel='Net Return_{t-1} (%)',
                  ylabel='Delta Omega_t (pp)',
                  title='Return Chasing: Do Good Returns Predict More Alternatives?')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_return_chasing.png')
plt.show()

## Section 4: Herd Behavior / Convergence

Do systems below the cross-sectional mean omega catch up faster? Galton regression toward the mean.

In [ ]:
# Convergence: deviation from cross-sectional mean -> subsequent change
# For each system-year, compute deviation from year mean
yearly_mean = sys_panel.groupby('fy')['omega'].mean().rename('omega_year_mean')
sys_panel = sys_panel.merge(yearly_mean, on='fy', how='left')
sys_panel['omega_dev'] = sys_panel['omega'] - sys_panel['omega_year_mean']
sys_panel['omega_dev_lag1'] = sys_panel.groupby('pub_id')['omega_dev'].shift(1)

# Galton regression: delta_omega_t = alpha + beta * omega_dev_{t-1}
galton_data = sys_panel.dropna(subset=['delta_omega', 'omega_dev_lag1']).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Galton regression
binscatter_simple(galton_data['omega_dev_lag1'], galton_data['delta_omega'],
                  ax=axes[0],
                  xlabel='Omega Deviation from Year Mean_{t-1} (pp)',
                  ylabel='Delta Omega_t (pp)',
                  title='Mean Reversion: Do Below-Average Systems Catch Up?')

# Right: Initial omega (2002) vs. long change (2002-2021)
sys_2002 = sys_df[sys_df['fy'] == 2002][['pub_id', 'omega']].rename(columns={'omega': 'omega_2002'})
sys_2021 = sys_df[sys_df['fy'] == 2021][['pub_id', 'omega']].rename(columns={'omega': 'omega_2021'})
long_change = sys_2002.merge(sys_2021, on='pub_id').dropna()
long_change['delta_omega_long'] = long_change['omega_2021'] - long_change['omega_2002']

binscatter_simple(long_change['omega_2002'], long_change['delta_omega_long'],
                  ax=axes[1],
                  xlabel='Omega in 2002 (%)',
                  ylabel='Delta Omega, 2002-2021 (pp)',
                  title='Long-Run Convergence: Early Adopters vs. Laggards')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_convergence.png')
plt.show()

## Section 5: Size Tiers

Do small pensions allocate differently from large ones?

In [ ]:
# Size tiers: tercile-based analysis
# Assign size tier based on median AUM across years for each system
sys_med_aum = sys_df.groupby('pub_id')['MktAssets_net'].median().reset_index()
sys_med_aum.columns = ['pub_id', 'median_aum']
sys_med_aum['size_tier'] = pd.qcut(sys_med_aum['median_aum'], 3, labels=['Small', 'Medium', 'Large'])
sys_panel_size = sys_df.merge(sys_med_aum[['pub_id', 'size_tier']], on='pub_id', how='left')

# Omega time series by size tier
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
tier_colors = {'Small': COLORS['green'], 'Medium': COLORS['blue'], 'Large': COLORS['orange']}

# Left: omega over time
ax = axes[0]
for tier in ['Small', 'Medium', 'Large']:
    ts = []
    for yr in range(2001, 2022):
        sub = sys_panel_size[(sys_panel_size['fy'] == yr) & (sys_panel_size['size_tier'] == tier)]
        vals = sub['omega'].dropna()
        if len(vals) > 0:
            ts.append({'fy': yr, 'median': vals.median() * 100})
    ts_df = pd.DataFrame(ts)
    ax.plot(ts_df['fy'], ts_df['median'], lw=2, marker='o', markersize=3,
            color=tier_colors[tier], label=tier)

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Median Omega (%)')
ax.set_title('Omega by Size Tier')
ax.legend()
ax.grid(alpha=0.3)

# Middle: alternatives share over time
ax = axes[1]
for tier in ['Small', 'Medium', 'Large']:
    ts = []
    for yr in range(2001, 2022):
        sub = sys_panel_size[(sys_panel_size['fy'] == yr) & (sys_panel_size['size_tier'] == tier)]
        vals = sub['alternatives_trgt'].dropna()
        if len(vals) > 0:
            ts.append({'fy': yr, 'median': vals.median() * 100})
    ts_df = pd.DataFrame(ts)
    ax.plot(ts_df['fy'], ts_df['median'], lw=2, marker='o', markersize=3,
            color=tier_colors[tier], label=tier)

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Median Alternatives Target (%)')
ax.set_title('Alternatives by Size Tier')
ax.legend()
ax.grid(alpha=0.3)

# Right: funding ratio over time
ax = axes[2]
for tier in ['Small', 'Medium', 'Large']:
    ts = []
    for yr in range(2001, 2022):
        sub = sys_panel_size[(sys_panel_size['fy'] == yr) & (sys_panel_size['size_tier'] == tier)]
        vals = sub['ActFundedRatio_GASB'].dropna()
        if len(vals) > 0:
            ts.append({'fy': yr, 'median': vals.median() * 100})
    ts_df = pd.DataFrame(ts)
    ax.plot(ts_df['fy'], ts_df['median'], lw=2, marker='o', markersize=3,
            color=tier_colors[tier], label=tier)

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Median Funding Ratio (%)')
ax.set_title('Funding by Size Tier')
ax.legend()
ax.grid(alpha=0.3)

fig.suptitle('Size Tier Analysis (Terciles by Median AUM)', fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_size_tiers.png')
plt.show()

# Print tier AUM cutoffs
for tier in ['Small', 'Medium', 'Large']:
    aum_vals = sys_med_aum.loc[sys_med_aum['size_tier'] == tier, 'median_aum']
    print(f'{tier}: N={len(aum_vals)}, AUM range: ${aum_vals.min()/1e9:.1f}B – ${aum_vals.max()/1e9:.1f}B')

## Section 6: Fiscal Year Timing Effects

Do June-FY vs. December-FY plans show different patterns?

In [ ]:
# Fiscal year timing: pfmonth from performance file
# Group into June-FY (month 6) vs December-FY (month 12) vs Other
sys_pfmonth = sys_df.dropna(subset=['pfmonth']).copy()
sys_pfmonth['fy_group'] = sys_pfmonth['pfmonth'].apply(
    lambda x: 'June FY' if x == 6 else ('December FY' if x == 12 else 'Other'))

# Distribution of FY end months
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Count by FY month
ax = axes[0]
month_counts = sys_pfmonth.groupby('pfmonth')['pub_id'].nunique().sort_index()
ax.bar(month_counts.index, month_counts.values, color=COLORS['blue'], alpha=0.85)
ax.set_xlabel('Fiscal Year End Month')
ax.set_ylabel('Number of Systems')
ax.set_title('FY End Month Distribution')
ax.set_xticks(range(1, 13))
ax.grid(axis='y', alpha=0.3)

# Middle: Performance by FY group
ax = axes[1]
for fy_group, color in [('June FY', COLORS['blue']), ('December FY', COLORS['orange']), ('Other', COLORS['green'])]:
    ts = []
    for yr in range(2001, 2022):
        sub = sys_pfmonth[(sys_pfmonth['fy'] == yr) & (sys_pfmonth['fy_group'] == fy_group)]
        vals = sub['ret_bgnassets'].dropna() * 100
        if len(vals) > 0:
            ts.append({'fy': yr, 'median': vals.median()})
    if ts:
        ts_df = pd.DataFrame(ts)
        ax.plot(ts_df['fy'], ts_df['median'], lw=2, marker='o', markersize=3,
                color=color, label=fy_group)

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Median Net Return (%)')
ax.set_title('Returns by FY End Month Group')
ax.legend()
ax.grid(alpha=0.3)
ax.axhline(0, color='black', lw=0.5)

# Right: Omega by FY group (2021)
ax = axes[2]
sys_2021_pfm = sys_pfmonth[sys_pfmonth['fy'] == 2021].dropna(subset=['omega'])
for fy_group, color in [('June FY', COLORS['blue']), ('December FY', COLORS['orange']), ('Other', COLORS['green'])]:
    vals = sys_2021_pfm.loc[sys_2021_pfm['fy_group'] == fy_group, 'omega'] * 100
    if len(vals) > 0:
        ax.hist(vals, bins=15, alpha=0.5, color=color, label=f'{fy_group} (N={len(vals)})', edgecolor='white')

ax.set_xlabel('Omega (%)')
ax.set_ylabel('Count')
ax.set_title('Omega Distribution by FY Group (2021)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_fy_timing.png')
plt.show()

print('Systems by FY end month group:')
print(sys_pfmonth.groupby('fy_group')['pub_id'].nunique().to_string())

## Section 7: Target-Actual Divergence Patterns

When targets and actuals diverge, what happens next year? Rebalancing dynamics.

In [ ]:
# Target-actual divergence and subsequent rebalancing
sys_panel['l_pt'] = sys_panel['risky_actl'] - sys_panel['risky_trgt']
sys_panel['l_pt_lag1'] = sys_panel.groupby('pub_id')['l_pt'].shift(1)
sys_panel['delta_risky_actl'] = sys_panel['risky_actl'] - sys_panel.groupby('pub_id')['risky_actl'].shift(1)

# Rebalancing: does overshooting targets predict a subsequent reduction in actuals?
rebal_data = sys_panel.dropna(subset=['delta_risky_actl', 'l_pt_lag1']).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Binscatter of rebalancing
binscatter_simple(rebal_data['l_pt_lag1'], rebal_data['delta_risky_actl'],
                  ax=axes[0],
                  xlabel='Actual-Target Gap_{t-1} (pp)',
                  ylabel='Delta Actual Risky_{t} (pp)',
                  title='Rebalancing: Does Overshooting Predict Correction?')

# Right: Distribution of actual-target gap by year
ax = axes[1]
gap_by_year = []
for yr in range(2001, 2022):
    vals = sys_panel.loc[sys_panel['fy'] == yr, 'l_pt'].dropna() * 100
    if len(vals) > 0:
        gap_by_year.append({
            'fy': yr, 'p25': vals.quantile(0.25), 'median': vals.median(),
            'p75': vals.quantile(0.75), 'pct_over': (vals > 0).mean() * 100,
        })

gby_df = pd.DataFrame(gap_by_year)
ax.bar(gby_df['fy'], gby_df['pct_over'], color=COLORS['blue'], alpha=0.7)
ax.axhline(50, color='red', ls='--', alpha=0.5)
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('% of Systems Over-Allocated to Risky')
ax.set_title('Share of Systems Above Risky Target')
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_rebalancing.png')
plt.show()

## Section 8: Maturity and Risk Appetite

Does the relationship between maturity (% retired) and omega change when conditioning on funding status?

In [ ]:
# Maturity × funding interaction
sys_2021_full = sys_df[sys_df['fy'] == 2021].dropna(subset=['omega', 'pct_retired', 'ActFundedRatio_GASB']).copy()

# Split by funding status
sys_2021_full['funding_group'] = pd.qcut(sys_2021_full['ActFundedRatio_GASB'], 2,
                                          labels=['Underfunded', 'Well-funded'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Overall scatter
ax = axes[0]
ax.scatter(sys_2021_full['pct_retired'] * 100, sys_2021_full['omega'] * 100,
           c=sys_2021_full['ActFundedRatio_GASB'] * 100, cmap='RdYlGn',
           s=40, alpha=0.7, edgecolor='white')
cb = plt.colorbar(ax.collections[0], ax=ax)
cb.set_label('Funding Ratio (%)')
# OLS line
X = sm.add_constant(sys_2021_full['pct_retired'])
model = sm.OLS(sys_2021_full['omega'], X).fit()
x_line = np.linspace(sys_2021_full['pct_retired'].min(), sys_2021_full['pct_retired'].max(), 100)
ax.plot(x_line * 100, (model.params['const'] + model.params['pct_retired'] * x_line) * 100,
        '--', color='red', lw=1.5)
ax.set_xlabel('Percent Retired (%)')
ax.set_ylabel('Omega (%)')
ax.set_title(f'Maturity vs. Omega (FY2021)\nbeta={model.params["pct_retired"]:.3f}, t={model.tvalues["pct_retired"]:.2f}')
ax.grid(alpha=0.3)

# Middle: Split by funding
ax = axes[1]
for group, color in [('Underfunded', COLORS['red']), ('Well-funded', COLORS['green'])]:
    sub = sys_2021_full[sys_2021_full['funding_group'] == group]
    ax.scatter(sub['pct_retired'] * 100, sub['omega'] * 100, color=color,
               s=40, alpha=0.6, label=group, edgecolor='white')
    # OLS
    X = sm.add_constant(sub['pct_retired'])
    m = sm.OLS(sub['omega'], X).fit()
    x_l = np.linspace(sub['pct_retired'].min(), sub['pct_retired'].max(), 50)
    ax.plot(x_l * 100, (m.params['const'] + m.params['pct_retired'] * x_l) * 100,
            '--', color=color, lw=1.5, alpha=0.8)

ax.set_xlabel('Percent Retired (%)')
ax.set_ylabel('Omega (%)')
ax.set_title('Maturity vs. Omega: Split by Funding')
ax.legend()
ax.grid(alpha=0.3)

# Right: Interaction regression (panel, all years)
panel_full = sys_df.dropna(subset=['omega', 'pct_retired', 'ActFundedRatio_GASB', 'StateAbbrev']).copy()
panel_full['pct_ret_x_funding'] = panel_full['pct_retired'] * panel_full['ActFundedRatio_GASB']

y = panel_full['omega']
X = sm.add_constant(panel_full[['pct_retired', 'ActFundedRatio_GASB', 'pct_ret_x_funding']])
model_int = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': panel_full['StateAbbrev']})

# Results table
ax = axes[2]
ax.axis('off')
results_text = (
    'Panel OLS: omega = a + b1*pct_retired + b2*funding + b3*interaction\n'
    '(State-clustered SE)\n\n'
    f'b1 (pct_retired):    {model_int.params["pct_retired"]:.4f}  (t={model_int.tvalues["pct_retired"]:.2f})\n'
    f'b2 (funding):        {model_int.params["ActFundedRatio_GASB"]:.4f}  (t={model_int.tvalues["ActFundedRatio_GASB"]:.2f})\n'
    f'b3 (interaction):    {model_int.params["pct_ret_x_funding"]:.4f}  (t={model_int.tvalues["pct_ret_x_funding"]:.2f})\n\n'
    f'R² = {model_int.rsquared:.3f}\n'
    f'N = {int(model_int.nobs)}'
)
ax.text(0.1, 0.9, results_text, transform=ax.transAxes, va='top', fontsize=10,
        family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.set_title('Interaction Regression')

fig.suptitle('Maturity, Funding, and Risk Appetite', fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ra_maturity_funding.png')
plt.show()